# The Price is Right

## Week 8 Order of Play

Day 1: Modal.com and SpecialistAgent  
Day 2: RAG, FrontierAgent, Ensemble Agent  
Day 3: ScannerAgent, MessengerAgent  
Day 4: AutonomousPlannerAgent and DealAgentFramework  
Day 5: The Price Is Right Finale


Today we'll build another piece of the puzzle: a ScanningAgent that looks for promising deals by subscribing to RSS feeds.

In [1]:
import os
from dotenv import load_dotenv
from openai import OpenAI
from agents.deals import ScrapedDeal, DealSelection
import logging
import requests
load_dotenv(override=True)
openai = OpenAI()
MODEL = 'gpt-5-mini'

In [2]:
deals = ScrapedDeal.fetch(show_progress=True)

100%|██████████| 3/3 [01:36<00:00, 32.19s/it]


In [3]:
len(deals)

30

In [4]:
deals[10].describe()

'Title: Dell Laptops & Chromebooks Deals at Woot for From $240 + free shipping w/ Prime\nDetails: Refurbished and factory-reconditioned Dell laptops are available starting at $240, with models ranging up to $1,050. A 90-Day Woot limited warranty is provided with purchase. The sale ends May 30. Buy Now at Woot! An Amazon Company\nFeatures: Models include Latitude 3450, 5330, 5420, 7430, 7455, Pro 14 Plus, Pro 16 Plus Screen sizes from 11.6" to 16" Configurations include 2-in-1 touchscreen and standard clamshell Condition: refurbished or factory reconditioned\nURL: https://www.dealnews.com/Dell-Laptops-Chromebooks-Deals-at-Woot-for-From-240-free-shipping-w-Prime/21833350.html?iref=rss-c39'

### We are going to ask GPT-5-mini to summarize deals and identify their price

In [7]:
SYSTEM_PROMPT = """You identify and summarize the 5 most detailed deals from a list, by selecting deals that have the most detailed, high quality description and the most clear price.
Respond strictly in JSON with no explanation, using this format. You should provide the price as a number derived from the description. If the price of a deal isn't clear, do not include that deal in your response.
Most important is that you respond with the 5 deals that have the most detailed product description with price. It's not important to mention the terms of the deal; most important is a thorough description of the product.
Be careful with products that are described as "$XXX off" or "reduced by $XXX" - this isn't the actual price of the product. Only respond with products when you are highly confident about the price. 
"""

USER_PROMPT_PREFIX = """Respond with the most promising 5 deals from this list, selecting those which have the most detailed, high quality product description and a clear price that is greater than 0.
You should rephrase the description to be a summary of the product itself, not the terms of the deal.
Remember to respond with a short paragraph of text in the product_description field for each of the 5 items that you select.
Be careful with products that are described as "$XXX off" or "reduced by $XXX" - this isn't the actual price of the product. Only respond with products when you are highly confident about the price. 

Deals:

"""

USER_PROMPT_SUFFIX = "\n\nInclude exactly 5 deals, no more."

In [8]:
# this makes a suitable user prompt given scraped deals

def make_user_prompt(scraped):
    user_prompt = USER_PROMPT_PREFIX
    user_prompt += '\n\n'.join([scrape.describe() for scrape in scraped])
    user_prompt += USER_PROMPT_SUFFIX
    return user_prompt

In [9]:
# Let's create a user prompt for the deals we just scraped, and look at how it begins

user_prompt = make_user_prompt(deals)
print(user_prompt[:2000])
messages = [{"role": "system", "content": SYSTEM_PROMPT}, {"role": "user", "content": user_prompt}]

Respond with the most promising 5 deals from this list, selecting those which have the most detailed, high quality product description and a clear price that is greater than 0.
You should rephrase the description to be a summary of the product itself, not the terms of the deal.
Remember to respond with a short paragraph of text in the product_description field for each of the 5 items that you select.
Be careful with products that are described as "$XXX off" or "reduced by $XXX" - this isn't the actual price of the product. Only respond with products when you are highly confident about the price. 

Deals:

Title: Charging Devices at Woot + free shipping w/ Prime
Details: This Woot Charging Devices Sale runs through May 31 and includes power banks, wireless chargers, surge protectors, cables, and USB hubs from brands including Anker and Baseus. Prime members get free standard shipping. Shop Now at Woot! An Amazon Company
Features: Power banks ranging from 10,000mAh to 50,000mAh Wireless 

In [10]:
response = openai.chat.completions.parse(model=MODEL, messages=messages, response_format=DealSelection, reasoning_effort="minimal")
results = response.choices[0].message.parsed
results

DealSelection(deals=[Deal(product_description="Refurbished Apple AirPods (3rd generation) with the Lightning charging case. These are refurbished to a 'Good' condition and include Bluetooth wireless connectivity and the standard in-ear fit of the 3rd-gen model. The listing includes a one-year warranty serviced by Allstate and comes with free 2–3 day delivery, making them a complete refurbished package for everyday audio use.", price=75.0, url='https://www.dealnews.com/Refurb-Apple-Air-Pods-w-Lightning-Charging-Case-3-rd-Gen-for-75-free-shipping/21833339.html?iref=rss-c142'), Deal(product_description='Belkin Slim 10,000mAh USB-C power bank with dual USB‑C ports and a digital battery display. It supports up to 20W output to a single device or 15W shared across two devices, includes TSA-compliant battery capacity for travel, and provides a more precise charge level readout than typical LED-dot indicators.', price=5.0, url='https://www.dealnews.com/Belkin-Slim-10-000-m-Ah-20-W-USB-C-Power-

In [11]:
for deal in results.deals:
    print(deal.product_description)
    print(deal.price)
    print(deal.url)
    print()


Refurbished Apple AirPods (3rd generation) with the Lightning charging case. These are refurbished to a 'Good' condition and include Bluetooth wireless connectivity and the standard in-ear fit of the 3rd-gen model. The listing includes a one-year warranty serviced by Allstate and comes with free 2–3 day delivery, making them a complete refurbished package for everyday audio use.
75.0
https://www.dealnews.com/Refurb-Apple-Air-Pods-w-Lightning-Charging-Case-3-rd-Gen-for-75-free-shipping/21833339.html?iref=rss-c142

Belkin Slim 10,000mAh USB-C power bank with dual USB‑C ports and a digital battery display. It supports up to 20W output to a single device or 15W shared across two devices, includes TSA-compliant battery capacity for travel, and provides a more precise charge level readout than typical LED-dot indicators.
5.0
https://www.dealnews.com/Belkin-Slim-10-000-m-Ah-20-W-USB-C-Power-Bank-free-shipping/21833322.html?iref=rss-c142

Anthbot Genie 3000 robot lawn mower with wire‑free RTK 

In [12]:
root = logging.getLogger()
root.setLevel(logging.INFO)

In [13]:
from agents.scanner_agent import ScannerAgent

In [14]:
agent = ScannerAgent()
result = agent.scan()

INFO:root:[Scanner Agent] Scanner Agent is initializing
INFO:root:[Scanner Agent] Scanner Agent is ready
INFO:root:[Scanner Agent] Scanner Agent is about to fetch deals from RSS feed
INFO:root:[Scanner Agent] Scanner Agent received 30 deals not already scraped
INFO:root:[Scanner Agent] Scanner Agent is calling OpenAI using Structured Outputs
INFO:httpx:HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
INFO:root:[Scanner Agent] Scanner Agent received 5 selected deals with price>0 from OpenAI


In [ ]:
result

### Introducing Pushover

Pushover is a nifty tool for sending Push Notifications to your phone.

It's super easy to set up and install!

Simply visit https://pushover.net/ and click 'Login or Signup' on the top right to sign up for a free account, and create your API keys.

Once you've signed up, on the home screen, click "Create an Application/API Token", and give it any name (like AIEngineer) and click Create Application.

Then add 2 lines to your `.env` file:

PUSHOVER_USER=_put the key that's on the top right of your Pushover home screen and probably starts with a u_  
PUSHOVER_TOKEN=_put the key when you click into your new application called Agents (or whatever) and probably starts with an a_

Remember to save your `.env` file, and run `load_dotenv(override=True)` after saving, to set your environment variables.

Finally, click "Add Phone, Tablet or Desktop" to install on your phone.

In [ ]:
load_dotenv(override=True)

In [ ]:
pushover_user = os.getenv('PUSHOVER_USER')
pushover_token = os.getenv('PUSHOVER_TOKEN')
pushover_url = "https://api.pushover.net/1/messages.json"

In [ ]:
if pushover_user:
    print(f"Pushover user found and starts with {pushover_user[0]}")
else:
    print("Pushover user not found")

if pushover_token:
    print(f"Pushover token found and starts with {pushover_token[0]}")
else:
    print("Pushover token not found")

In [ ]:
def push(message):
    print(f"Push: {message}")
    payload = {"user": pushover_user, "token": pushover_token, "message": message}
    requests.post(pushover_url, data=payload)

In [ ]:
push("MASSIVE DEAL!!")

In [ ]:
from agents.messaging_agent import MessagingAgent

agent = MessagingAgent()
agent.push("SUCH A MASSIVE DEAL!!")

In [ ]:
agent.notify("A special deal on Sumsung 60 inch LED TV going at a great bargain", 300, 1000, "www.samsung.com")